In [0]:
# =============================================================
# CELDA 1: GOLD — KPIs + Resumen de Mercado Robusto
# =============================================================
import json
import boto3
import re
import unicodedata
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Credenciales: Databricks secrets (producción) → archivo local (desarrollo)
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
    }
except Exception:
    try:
        with open("aws_secrets.json", "r") as f:
            config = json.load(f)
    except FileNotFoundError:
        raise SystemExit("❌ Credenciales no disponibles. Configura Databricks Secrets (scope='aws') o coloca aws_secrets.json.")

AWS_KEY = config["aws_access_key"]
AWS_SECRET = config["aws_secret_key"]
BUCKET = "bronce-scrap-date"
MIN_ANALYTICS_OBS = {
    "city": 15,
    "comuna": 12,
    "sector": 8,
}

S3_OPTS = {
    "fs.s3a.access.key": AWS_KEY,
    "fs.s3a.secret.key": AWS_SECRET,
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

s3_check = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1",
)

def normalize_text(value):
    text = "" if value is None else str(value)
    text = text.lower().strip()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\|.*", "", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

COMUNA_KEYWORDS_BY_CITY = {
    "medellin": {
        "poblado": [
            "poblado", "provenza", "manila", "castropol", "lalinde",
            "aguacatala", "patio bonito", "los balsos", "el tesoro",
            "santa maria de los angeles",
        ],
        "laureles_estadio": [
            "laureles", "estadio", "la america", "america", "floresta",
            "calasanz", "suramericana", "velodromo", "los colores",
            "simon bolivar", "carlos e restrepo",
        ],
        "belen": [
            "belen", "fatima", "rosales", "granada", "loma de los bernal",
            "nogal", "aliadas", "mira valle",
        ],
        "robledo": [
            "robledo", "pajarito", "pilarica", "cordoba", "villa flora",
            "aures", "altamira",
        ],
        "castilla_doce_octubre": [
            "castilla", "doce de octubre", "pedregal", "tejelo", "girardot",
            "florencia", "boyaca",
        ],
        "aranjuez_manrique": [
            "aranjuez", "manrique", "berlin", "campo valdes", "palos verdes",
            "miraflores",
        ],
        "popular_santa_cruz": [
            "popular", "santa cruz", "andalucia", "moscu", "villa guadalupe",
        ],
        "buenos_aires": [
            "buenos aires", "bombona", "la milagrosa", "barcelona", "gerona",
        ],
        "guayabal": [
            "guayabal", "trinidad", "cristo rey", "santa fe",
        ],
        "centro": [
            "candelaria", "centro", "prado", "sevilla", "san diego",
        ],
    },
    "bogota": {
        "usaquen": ["usaquen", "santa barbara", "cedritos", "bella suiza", "country"],
        "chapinero": ["chapinero", "rosales", "nogal", "chico", "castellana", "quinta camacho"],
        "suba": ["suba", "niza", "colina", "gratamira", "mazuren"],
        "teusaquillo_barrios_unidos": ["teusaquillo", "salitre", "parque simon bolivar", "barrios unidos"],
        "kennedy_fontibon": ["kennedy", "fontibon", "hayuelos", "modelia", "castilla"],
    },
}

CITY_ALIAS_TO_TOKEN = {
    "bogota": ["bogota", "bogota dc", "bogota d c", "bogota distrito capital"],
    "medellin": ["medellin", "medallo"],
    "cali": ["cali", "santiago de cali"],
    "barranquilla": ["barranquilla", "bquilla"],
    "cartagena": ["cartagena", "cartagena de indias"],
    "bucaramanga": ["bucaramanga"],
    "pereira": ["pereira"],
    "manizales": ["manizales"],
    "cucuta": ["cucuta", "san jose de cucuta"],
    "santa marta": ["santa marta"],
}

SECTOR_STOPWORDS = {
    "apartamento", "apartaestudio", "casa", "lote", "finca", "oficina",
    "local", "comercial", "venta", "arriendo", "colombia", "sector",
    "barrio", "zona", "urbanizacion", "unidad", "conjunto",
}
CITY_ALIAS_TOKENS = {token for aliases in CITY_ALIAS_TO_TOKEN.values() for alias in aliases for token in alias.split()}
CITY_ALIAS_TOKENS.update(CITY_ALIAS_TO_TOKEN.keys())

def canonicalize_city_token(raw_city_token):
    city_value = normalize_text(raw_city_token)
    if not city_value or city_value == "otra_ciudad":
        return "otra_ciudad"
    for city_token, aliases in CITY_ALIAS_TO_TOKEN.items():
        if city_value == city_token or any(alias in city_value for alias in aliases):
            return city_token
    return city_value

def assign_comuna(city_token, ubicacion_limpia):
    city = normalize_text(city_token)
    location = normalize_text(ubicacion_limpia)
    city_map = COMUNA_KEYWORDS_BY_CITY.get(city, {})
    for comuna_name, keywords in city_map.items():
        if any(keyword in location for keyword in keywords):
            return comuna_name
    return "comuna_otra"

def extract_sector_mercado(city_token, comuna_mercado, ubicacion_limpia):
    location = normalize_text(ubicacion_limpia)
    if not location:
        return comuna_mercado if comuna_mercado != "comuna_otra" else "sector_otra"

    city_map = COMUNA_KEYWORDS_BY_CITY.get(normalize_text(city_token), {})
    matched_keywords = set()
    for keywords in city_map.values():
        for keyword in keywords:
            if keyword in location:
                matched_keywords.update(keyword.split())

    sector_tokens = []
    for token in location.split():
        if token in SECTOR_STOPWORDS or token in CITY_ALIAS_TOKENS or token in matched_keywords:
            continue
        sector_tokens.append(token)
        if len(sector_tokens) >= 3:
            break

    sector_name = " ".join(sector_tokens).strip()
    if not sector_name and comuna_mercado != "comuna_otra":
        sector_name = comuna_mercado.replace("_", " ")
    return sector_name or "sector_otra"

def build_market_analytics(df_input, level_name, group_cols, min_obs):
    analytics = (
        df_input.groupBy(*(group_cols + ["tipo_inmueble"]))
        .agg(
            F.count("*").alias("market_n"),
            F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano"),
            F.expr("percentile_approx(area_m2, array(0.25, 0.5, 0.75))").alias("area_quantiles"),
            F.expr("percentile_approx(precio_m2, array(0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95))").alias("precio_m2_quantiles"),
            F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
            F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
            F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
        )
        .filter(F.col("market_n") >= min_obs)
        .withColumn("analytics_level", F.lit(level_name))
    )

    if "city_token" not in group_cols:
        analytics = analytics.withColumn("city_token", F.lit("otra_ciudad"))
    if "comuna_mercado" not in group_cols:
        analytics = analytics.withColumn("comuna_mercado", F.lit("comuna_otra"))
    if "sector_mercado" not in group_cols:
        analytics = analytics.withColumn("sector_mercado", F.lit("sector_otra"))

    analytics = (
        analytics
        .withColumn("area_p25", F.col("area_quantiles").getItem(0))
        .withColumn("area_mediana", F.col("area_quantiles").getItem(1))
        .withColumn("area_p75", F.col("area_quantiles").getItem(2))
        .withColumn("precio_m2_p05", F.col("precio_m2_quantiles").getItem(0))
        .withColumn("precio_m2_p10", F.col("precio_m2_quantiles").getItem(1))
        .withColumn("precio_m2_p25", F.col("precio_m2_quantiles").getItem(2))
        .withColumn("precio_m2_mediano", F.col("precio_m2_quantiles").getItem(3))
        .withColumn("precio_m2_p75", F.col("precio_m2_quantiles").getItem(4))
        .withColumn("precio_m2_p90", F.col("precio_m2_quantiles").getItem(5))
        .withColumn("precio_m2_p95", F.col("precio_m2_quantiles").getItem(6))
        .withColumn("precio_m2_iqr", F.col("precio_m2_p75") - F.col("precio_m2_p25"))
        .withColumn("lower_bound_ref", F.greatest(F.lit(0.0), F.col("precio_m2_p10") - F.col("precio_m2_iqr") * F.lit(1.5)))
        .withColumn("upper_bound_ref", F.col("precio_m2_p90") + F.col("precio_m2_iqr") * F.lit(1.5))
        .withColumn(
            "market_quality_score",
            F.round(
                F.least(
                    F.lit(100.0),
                    (F.log1p(F.col("market_n")) / F.log(F.lit(201.0))) * F.lit(60.0)
                    + F.coalesce(F.col("pct_multiportal"), F.lit(0.0)) * F.lit(0.25)
                    + (F.lit(100.0) - F.least(F.coalesce(F.col("dispersion_promedio_pct"), F.lit(100.0)), F.lit(100.0))) * F.lit(0.15),
                ),
                1,
            ),
        )
        .withColumn(
            "market_key",
            F.concat_ws("__", "analytics_level", "city_token", "comuna_mercado", "sector_mercado", "tipo_inmueble"),
        )
        .drop("area_quantiles", "precio_m2_quantiles", "precio_m2_iqr")
    )

    return analytics.select(
        "analytics_level",
        "market_key",
        "city_token",
        "comuna_mercado",
        "sector_mercado",
        "tipo_inmueble",
        "market_n",
        "precio_mediano",
        "area_p25",
        "area_mediana",
        "area_p75",
        "precio_m2_p05",
        "precio_m2_p10",
        "precio_m2_p25",
        "precio_m2_mediano",
        "precio_m2_p75",
        "precio_m2_p90",
        "precio_m2_p95",
        "lower_bound_ref",
        "upper_bound_ref",
        "promedio_portales",
        "dispersion_promedio_pct",
        "pct_multiportal",
        "market_quality_score",
    )

normalize_text_udf = F.udf(normalize_text, StringType())
canonicalize_city_udf = F.udf(canonicalize_city_token, StringType())
assign_comuna_udf = F.udf(assign_comuna, StringType())
extract_sector_udf = F.udf(extract_sector_mercado, StringType())

# --- Leer Silver Deduped (output de 02b_Dedup_Cross_Portal) ---
# Fallback a master_inmuebles si master_deduped no existe aún
try:
    s3_check.list_objects_v2(
        Bucket=BUCKET, Prefix="silver/master_deduped/_delta_log/", MaxKeys=1
    )["Contents"]
    ruta_silver = f"s3a://{BUCKET}/silver/master_deduped/"
    print("📖 Leyendo Silver DEDUPED (cross-portal)")
except (KeyError, Exception):
    ruta_silver = f"s3a://{BUCKET}/silver/master_inmuebles/"
    print("📖 Leyendo Silver original (dedup no ejecutado aún)")

reader = spark.read.format("delta")
for k, v in S3_OPTS.items():
    reader = reader.option(k, v)
df_silver = reader.load(ruta_silver)

print(f"   {df_silver.count()} registros")
print(f"   Columnas: {df_silver.columns}")

# --- Leer price intelligence si existe para enriquecer el resumen ---
try:
    s3_check.list_objects_v2(
        Bucket=BUCKET, Prefix="gold/price_intelligence/_delta_log/", MaxKeys=1
    )["Contents"]
    reader_intel = spark.read.format("delta")
    for k, v in S3_OPTS.items():
        reader_intel = reader_intel.option(k, v)
    df_intel = reader_intel.load(f"s3a://{BUCKET}/gold/price_intelligence/")
    print("💰 Price intelligence disponible para enriquecer Gold")
except (KeyError, Exception):
    df_intel = None
    print("ℹ️ Price intelligence no disponible todavía")

# --- KPIs robustos por inmueble ---
df_gold_base = (
    df_silver
    .filter(
        F.col("precio_num").isNotNull() &
        (F.col("precio_num") > 0) &
        F.col("area_m2").isNotNull() &
        (F.col("area_m2") >= 20) &
        (F.col("area_m2") <= 800)
    )
    .withColumn("precio_m2", F.round(F.col("precio_num") / F.col("area_m2"), 0))
    .withColumn("city_token", canonicalize_city_udf(F.coalesce(F.col("city_token"), F.lit("otra_ciudad"))))
    .withColumn("tipo_inmueble", F.coalesce(F.col("tipo_inmueble"), F.lit("otro")))
    .withColumn(
        "zona_mercado",
        F.coalesce(
            F.when(F.length(F.col("ubicacion_norm")) > 4, F.col("ubicacion_norm")),
            F.when(F.length(F.col("ubicacion_raw")) > 4, F.lower(F.trim(F.col("ubicacion_raw")))),
            F.concat_ws(" | ", F.col("city_token"), F.col("tipo_inmueble"))
        )
    )
    .withColumn("ubicacion_limpia", normalize_text_udf(F.col("zona_mercado")))
    .withColumn("comuna_mercado", assign_comuna_udf(F.col("city_token"), F.col("ubicacion_limpia")))
    .withColumn("sector_mercado_raw", extract_sector_udf(F.col("city_token"), F.col("comuna_mercado"), F.col("ubicacion_limpia")))
    .withColumn(
        "market_segment",
        F.concat_ws("__", F.col("city_token"), F.col("tipo_inmueble"))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
)

# Filtro robusto por precio_m2 dentro de segmento ciudad + tipo
segment_stats = (
    df_gold_base
    .groupBy("market_segment")
    .agg(
        F.count("*").alias("segment_n"),
        F.expr("percentile_approx(precio_m2, 0.05)").alias("pm2_p05_segment"),
        F.expr("percentile_approx(precio_m2, 0.95)").alias("pm2_p95_segment"),
    )
)

global_pm2 = df_gold_base.agg(
    F.expr("percentile_approx(precio_m2, 0.02)").alias("pm2_p02_global"),
    F.expr("percentile_approx(precio_m2, 0.98)").alias("pm2_p98_global"),
).first()

df_gold = (
    df_gold_base
    .join(segment_stats, "market_segment", "left")
    .withColumn(
        "pm2_lower_bound",
        F.when(F.col("segment_n") >= 15, F.col("pm2_p05_segment")).otherwise(F.lit(global_pm2["pm2_p02_global"]))
    )
    .withColumn(
        "pm2_upper_bound",
        F.when(F.col("segment_n") >= 15, F.col("pm2_p95_segment")).otherwise(F.lit(global_pm2["pm2_p98_global"]))
    )
    .filter(F.col("precio_m2").between(F.col("pm2_lower_bound"), F.col("pm2_upper_bound")))
    .drop("pm2_lower_bound", "pm2_upper_bound", "pm2_p05_segment", "pm2_p95_segment")
)

sector_counts = (
    df_gold
    .groupBy("city_token", "sector_mercado_raw")
    .agg(F.count("*").alias("sector_n_raw"))
)

df_gold = (
    df_gold
    .join(sector_counts, ["city_token", "sector_mercado_raw"], "left")
    .withColumn(
        "sector_mercado",
        F.when(F.coalesce(F.col("sector_n_raw"), F.lit(0)) >= 12, F.col("sector_mercado_raw"))
         .when(F.col("comuna_mercado") != F.lit("comuna_otra"), F.col("comuna_mercado"))
         .otherwise(F.lit("sector_otra"))
    )
    .drop("sector_mercado_raw", "sector_n_raw")
)

print(f"\n📊 Gold detalle robusto: {df_gold.count()} inmuebles con KPI")
display(
    df_gold.select(
        "titulo", "city_token", "comuna_mercado", "sector_mercado", "zona_mercado", "tipo_inmueble",
        "precio_num", "area_m2", "precio_m2",
        "habitaciones", "banos", "garajes", "num_portales", "dispersion_pct_grupo"
    ).limit(10)
)

# --- Resumen Agregado por zona normalizada ---
df_resumen = (
    df_gold.groupBy("city_token", "zona_mercado", "tipo_inmueble")
    .agg(
        F.count("id_original").alias("total_ofertas"),
        F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano"),
        F.expr("percentile_approx(area_m2, 0.5)").alias("area_mediana"),
        F.expr("percentile_approx(precio_m2, 0.5)").alias("valor_m2_mediano"),
        F.expr("percentile_approx(precio_m2, 0.25)").alias("valor_m2_p25"),
        F.expr("percentile_approx(precio_m2, 0.75)").alias("valor_m2_p75"),
        F.round(F.avg("habitaciones"), 1).alias("habitaciones_promedio"),
        F.round(F.avg("banos"), 1).alias("banos_promedio"),
        F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
        F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
        F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
    )
    .filter(F.col("total_ofertas") >= 3)
)

df_sectorial = (
    df_gold.groupBy("city_token", "comuna_mercado", "sector_mercado", "tipo_inmueble")
    .agg(
        F.count("*").alias("sector_n"),
        F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano_sector"),
        F.expr("percentile_approx(area_m2, 0.5)").alias("area_mediana_sector"),
        F.expr("percentile_approx(precio_m2, 0.10)").alias("precio_m2_p10_sector"),
        F.expr("percentile_approx(precio_m2, 0.25)").alias("precio_m2_p25_sector"),
        F.expr("percentile_approx(precio_m2, 0.5)").alias("precio_m2_mediano_sector"),
        F.expr("percentile_approx(precio_m2, 0.75)").alias("precio_m2_p75_sector"),
        F.expr("percentile_approx(precio_m2, 0.90)").alias("precio_m2_p90_sector"),
        F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
        F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
        F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
    )
    .filter(F.col("sector_n") >= MIN_ANALYTICS_OBS["sector"])
    .withColumn("precio_m2_min_ref", F.col("precio_m2_p10_sector"))
    .withColumn("precio_m2_max_ref", F.col("precio_m2_p90_sector"))
    .orderBy(F.desc("sector_n"), F.desc("precio_m2_mediano_sector"))
)

df_market_analytics = (
    build_market_analytics(df_gold, "city", ["city_token"], MIN_ANALYTICS_OBS["city"])
    .unionByName(build_market_analytics(df_gold, "comuna", ["city_token", "comuna_mercado"], MIN_ANALYTICS_OBS["comuna"]))
    .unionByName(build_market_analytics(df_gold, "sector", ["city_token", "comuna_mercado", "sector_mercado"], MIN_ANALYTICS_OBS["sector"]))
    .orderBy(F.desc("market_quality_score"), F.desc("market_n"))
)

if df_intel is not None:
    df_oportunidades = (
        df_intel
        .withColumn("city_token", F.coalesce(F.col("city_token"), F.lit("otra_ciudad")))
        .withColumn("tipo_inmueble", F.coalesce(F.col("tipo_inmueble"), F.lit("otro")))
        .withColumn(
            "zona_mercado",
            F.coalesce(
                F.when(F.length(F.col("ubicacion_norm")) > 4, F.col("ubicacion_norm")),
                F.when(F.length(F.col("ubicacion_raw")) > 4, F.lower(F.trim(F.col("ubicacion_raw")))),
                F.concat_ws(" | ", F.col("city_token"), F.col("tipo_inmueble"))
            )
        )
        .groupBy("city_token", "zona_mercado", "tipo_inmueble")
        .agg(
            F.count("*").alias("oportunidades_cross_portal"),
            F.round(F.avg("ahorro_potencial"), 0).alias("ahorro_promedio_cross_portal"),
            F.round(F.avg("ahorro_pct"), 1).alias("ahorro_pct_promedio"),
        )
    )
    df_resumen = df_resumen.join(df_oportunidades, ["city_token", "zona_mercado", "tipo_inmueble"], "left")

df_resumen = df_resumen.orderBy(F.desc("valor_m2_mediano"), F.desc("total_ofertas"))

print(f"\n🏆 Zonas con >= 3 ofertas limpias: {df_resumen.count()}")
display(df_resumen.limit(100))

print(f"\n🧭 Sectores con >= {MIN_ANALYTICS_OBS['sector']} ofertas limpias: {df_sectorial.count()}")
display(df_sectorial.limit(100))

print(f"\n📐 Analítica jerárquica de mercado: {df_market_analytics.count()} filas")
display(df_market_analytics.limit(100))

In [ ]:
# =============================================================
# CELDA 2: AJUSTE GEOGRÁFICO — market_token como eje comercial
# =============================================================
import os
import sys
import importlib
import importlib.util

_geo_candidates = []
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _geo_candidates.append("/Workspace" + str(_nb_path).rsplit("/", 1)[0])
except Exception:
    pass
_vsc_file = globals().get("__vsc_ipynb_file__", "")
if _vsc_file:
    _geo_candidates.append(os.path.dirname(os.path.abspath(_vsc_file)))
_geo_candidates.append(os.getcwd())
for _candidate in _geo_candidates:
    if _candidate and _candidate not in sys.path:
        sys.path.insert(0, _candidate)

def _load_geo_module(module_name):
    try:
        return importlib.reload(importlib.import_module(module_name))
    except Exception as import_exc:
        last_error = import_exc
        for candidate_dir in _geo_candidates:
            if not candidate_dir:
                continue
            module_path = os.path.join(candidate_dir, f"{module_name}.py")
            if not os.path.isfile(module_path):
                continue
            try:
                spec = importlib.util.spec_from_file_location(module_name, module_path)
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                sys.modules[module_name] = module
                return module
            except Exception as file_exc:
                last_error = file_exc
        raise last_error

try:
    _geo_module = _load_geo_module("geography_catalog")
    GOLD_CITY_TO_MARKET = _geo_module.CITY_TO_MARKET
    print("✅ Gold usando geography_catalog.py para market_token.")
except Exception as exc:
    GOLD_CITY_TO_MARKET = {}
    print(f"⚠️ Gold no pudo cargar geography_catalog.py; usando market_token actual/fallback. Detalle: {str(exc)[:160]}")

def map_gold_market(city_token):
    city_value = normalize_text(city_token)
    if not city_value or city_value == "otra_ciudad":
        return "mercado_otro"
    return GOLD_CITY_TO_MARKET.get(city_value, "mercado_otro")

map_gold_market_udf = F.udf(map_gold_market, StringType())

def build_market_analytics_v2(df_input, level_name, group_cols, min_obs):
    analytics = (
        df_input.groupBy(*(group_cols + ["tipo_inmueble"]))
        .agg(
            F.count("*").alias("market_n"),
            F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano"),
            F.expr("percentile_approx(area_m2, array(0.25, 0.5, 0.75))").alias("area_quantiles"),
            F.expr("percentile_approx(precio_m2, array(0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95))").alias("precio_m2_quantiles"),
            F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
            F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
            F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
        )
        .filter(F.col("market_n") >= min_obs)
        .withColumn("analytics_level", F.lit(level_name))
    )

    if "market_token" not in group_cols:
        analytics = analytics.withColumn("market_token", F.lit("mercado_otro"))
    if "city_token" not in group_cols:
        analytics = analytics.withColumn("city_token", F.lit("otra_ciudad"))
    if "comuna_mercado" not in group_cols:
        analytics = analytics.withColumn("comuna_mercado", F.lit("comuna_otra"))
    if "sector_mercado" not in group_cols:
        analytics = analytics.withColumn("sector_mercado", F.lit("sector_otra"))

    analytics = (
        analytics
        .withColumn("area_p25", F.col("area_quantiles").getItem(0))
        .withColumn("area_mediana", F.col("area_quantiles").getItem(1))
        .withColumn("area_p75", F.col("area_quantiles").getItem(2))
        .withColumn("precio_m2_p05", F.col("precio_m2_quantiles").getItem(0))
        .withColumn("precio_m2_p10", F.col("precio_m2_quantiles").getItem(1))
        .withColumn("precio_m2_p25", F.col("precio_m2_quantiles").getItem(2))
        .withColumn("precio_m2_mediano", F.col("precio_m2_quantiles").getItem(3))
        .withColumn("precio_m2_p75", F.col("precio_m2_quantiles").getItem(4))
        .withColumn("precio_m2_p90", F.col("precio_m2_quantiles").getItem(5))
        .withColumn("precio_m2_p95", F.col("precio_m2_quantiles").getItem(6))
        .withColumn("precio_m2_iqr", F.col("precio_m2_p75") - F.col("precio_m2_p25"))
        .withColumn("lower_bound_ref", F.greatest(F.lit(0.0), F.col("precio_m2_p10") - F.col("precio_m2_iqr") * F.lit(1.5)))
        .withColumn("upper_bound_ref", F.col("precio_m2_p90") + F.col("precio_m2_iqr") * F.lit(1.5))
        .withColumn(
            "market_quality_score",
            F.round(
                F.least(
                    F.lit(100.0),
                    (F.log1p(F.col("market_n")) / F.log(F.lit(201.0))) * F.lit(60.0)
                    + F.coalesce(F.col("pct_multiportal"), F.lit(0.0)) * F.lit(0.25)
                    + (F.lit(100.0) - F.least(F.coalesce(F.col("dispersion_promedio_pct"), F.lit(100.0)), F.lit(100.0))) * F.lit(0.15),
                ),
                1,
            ),
        )
        .withColumn(
            "market_key",
            F.concat_ws("__", "analytics_level", "market_token", "city_token", "comuna_mercado", "sector_mercado", "tipo_inmueble"),
        )
        .drop("area_quantiles", "precio_m2_quantiles", "precio_m2_iqr")
    )

    return analytics.select(
        "analytics_level",
        "market_key",
        "market_token",
        "city_token",
        "comuna_mercado",
        "sector_mercado",
        "tipo_inmueble",
        "market_n",
        "precio_mediano",
        "area_p25",
        "area_mediana",
        "area_p75",
        "precio_m2_p05",
        "precio_m2_p10",
        "precio_m2_p25",
        "precio_m2_mediano",
        "precio_m2_p75",
        "precio_m2_p90",
        "precio_m2_p95",
        "lower_bound_ref",
        "upper_bound_ref",
        "promedio_portales",
        "dispersion_promedio_pct",
        "pct_multiportal",
        "market_quality_score",
    )

df_gold = (
    df_gold
    .withColumn("market_token", F.coalesce(F.col("market_token"), map_gold_market_udf(F.col("city_token"))))
    .withColumn(
        "market_token",
        F.when(F.length(F.trim(F.col("market_token"))) > 0, F.col("market_token")).otherwise(F.lit("mercado_otro"))
    )
    .withColumn("market_segment", F.concat_ws("__", F.col("market_token"), F.col("tipo_inmueble")))
)

if df_intel is not None:
    df_intel = (
        df_intel
        .withColumn("market_token", F.coalesce(F.col("market_token"), map_gold_market_udf(F.col("city_token"))))
        .withColumn(
            "market_token",
            F.when(F.length(F.trim(F.col("market_token"))) > 0, F.col("market_token")).otherwise(F.lit("mercado_otro"))
        )
    )

df_resumen = (
    df_gold.groupBy("market_token", "city_token", "zona_mercado", "tipo_inmueble")
    .agg(
        F.count("id_original").alias("total_ofertas"),
        F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano"),
        F.expr("percentile_approx(area_m2, 0.5)").alias("area_mediana"),
        F.expr("percentile_approx(precio_m2, 0.5)").alias("valor_m2_mediano"),
        F.expr("percentile_approx(precio_m2, 0.25)").alias("valor_m2_p25"),
        F.expr("percentile_approx(precio_m2, 0.75)").alias("valor_m2_p75"),
        F.round(F.avg("habitaciones"), 1).alias("habitaciones_promedio"),
        F.round(F.avg("banos"), 1).alias("banos_promedio"),
        F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
        F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
        F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
    )
    .filter(F.col("total_ofertas") >= 3)
)

df_sectorial = (
    df_gold.groupBy("market_token", "city_token", "comuna_mercado", "sector_mercado", "tipo_inmueble")
    .agg(
        F.count("*").alias("sector_n"),
        F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano_sector"),
        F.expr("percentile_approx(area_m2, 0.5)").alias("area_mediana_sector"),
        F.expr("percentile_approx(precio_m2, 0.10)").alias("precio_m2_p10_sector"),
        F.expr("percentile_approx(precio_m2, 0.25)").alias("precio_m2_p25_sector"),
        F.expr("percentile_approx(precio_m2, 0.5)").alias("precio_m2_mediano_sector"),
        F.expr("percentile_approx(precio_m2, 0.75)").alias("precio_m2_p75_sector"),
        F.expr("percentile_approx(precio_m2, 0.90)").alias("precio_m2_p90_sector"),
        F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
        F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
        F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
    )
    .filter(F.col("sector_n") >= MIN_ANALYTICS_OBS["sector"])
    .withColumn("precio_m2_min_ref", F.col("precio_m2_p10_sector"))
    .withColumn("precio_m2_max_ref", F.col("precio_m2_p90_sector"))
    .orderBy(F.desc("sector_n"), F.desc("precio_m2_mediano_sector"))
)

df_market_analytics = (
    build_market_analytics_v2(df_gold, "market", ["market_token"], max(MIN_ANALYTICS_OBS["city"], 20))
    .unionByName(build_market_analytics_v2(df_gold, "city", ["market_token", "city_token"], MIN_ANALYTICS_OBS["city"]))
    .unionByName(build_market_analytics_v2(df_gold, "comuna", ["market_token", "city_token", "comuna_mercado"], MIN_ANALYTICS_OBS["comuna"]))
    .unionByName(build_market_analytics_v2(df_gold, "sector", ["market_token", "city_token", "comuna_mercado", "sector_mercado"], MIN_ANALYTICS_OBS["sector"]))
    .orderBy(F.desc("market_quality_score"), F.desc("market_n"))
)

if df_intel is not None:
    df_oportunidades = (
        df_intel
        .withColumn("city_token", F.coalesce(F.col("city_token"), F.lit("otra_ciudad")))
        .withColumn("market_token", F.coalesce(F.col("market_token"), map_gold_market_udf(F.col("city_token"))))
        .withColumn("tipo_inmueble", F.coalesce(F.col("tipo_inmueble"), F.lit("otro")))
        .withColumn(
            "zona_mercado",
            F.coalesce(
                F.when(F.length(F.col("ubicacion_norm")) > 4, F.col("ubicacion_norm")),
                F.when(F.length(F.col("ubicacion_raw")) > 4, F.lower(F.trim(F.col("ubicacion_raw")))),
                F.concat_ws(" | ", F.col("city_token"), F.col("tipo_inmueble"))
            )
        )
        .groupBy("market_token", "city_token", "zona_mercado", "tipo_inmueble")
        .agg(
            F.count("*").alias("oportunidades_cross_portal"),
            F.round(F.avg("ahorro_potencial"), 0).alias("ahorro_promedio_cross_portal"),
            F.round(F.avg("ahorro_pct"), 1).alias("ahorro_pct_promedio"),
        )
    )
    df_resumen = df_resumen.join(df_oportunidades, ["market_token", "city_token", "zona_mercado", "tipo_inmueble"], "left")

df_resumen = df_resumen.orderBy(F.desc("valor_m2_mediano"), F.desc("total_ofertas"))

print("\n🧭 Ajuste market_token aplicado en Gold")
print(f"   market_token únicos en detalle: {df_gold.select('market_token').distinct().count():,}")
print(f"   mercado_resumen filas: {df_resumen.count():,}")
print(f"   mercado_sectorial filas: {df_sectorial.count():,}")
print(f"   mercado_analitica filas: {df_market_analytics.count():,}")
display(df_gold.select("market_token", "city_token", "comuna_mercado", "sector_mercado", "tipo_inmueble", "precio_m2").limit(20))

In [ ]:
# =============================================================
# CELDA 2: ANALÍTICA OPERATIVA POR PORTAL — Checkpoints + Señales Gold
# =============================================================
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, LongType, StructField, StructType, TimestampType

SILVER_PORTAL_CHECKPOINTS_PATH = config.get("silver_portal_checkpoints_path") or f"s3a://{BUCKET}/silver/portal_checkpoints/"

def load_silver_checkpoint_table(path):
    last_error = None
    for fmt in ["delta", "parquet"]:
        try:
            reader = spark.read.format(fmt)
            for k, v in S3_OPTS.items():
                reader = reader.option(k, v)
            return reader.load(path), fmt
        except Exception as exc:
            last_error = exc
    raise last_error

checkpoint_schema = StructType([
    StructField("portal", StringType(), True),
    StructField("checkpoint_key", StringType(), True),
    StructField("checkpoint_source_path", StringType(), True),
    StructField("checkpoint_activo", IntegerType(), True),
    StructField("checkpoint_last_page", LongType(), True),
    StructField("checkpoint_total_scraped", LongType(), True),
    StructField("checkpoint_updated_at", TimestampType(), True),
    StructField("checkpoint_read_error", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
])

try:
    df_checkpoints_silver, checkpoint_format = load_silver_checkpoint_table(SILVER_PORTAL_CHECKPOINTS_PATH)
    checkpoint_source_label = f"{SILVER_PORTAL_CHECKPOINTS_PATH} ({checkpoint_format})"
    print(f"🛰️ Leyendo Silver checkpoints ({checkpoint_format.upper()}): {SILVER_PORTAL_CHECKPOINTS_PATH}")
except Exception as e:
    checkpoint_source_label = ""
    df_checkpoints_silver = spark.createDataFrame([], schema=checkpoint_schema)
    print(f"ℹ️ Silver checkpoints no disponible en esta corrida: {str(e)[:220]}")

for col_name, col_type in [
    ("portal", StringType()),
    ("checkpoint_key", StringType()),
    ("checkpoint_source_path", StringType()),
    ("checkpoint_activo", IntegerType()),
    ("checkpoint_last_page", LongType()),
    ("checkpoint_total_scraped", LongType()),
    ("checkpoint_updated_at", TimestampType()),
    ("checkpoint_read_error", StringType()),
    ("ingestion_timestamp", TimestampType()),
]:
    if col_name not in df_checkpoints_silver.columns:
        df_checkpoints_silver = df_checkpoints_silver.withColumn(col_name, F.lit(None).cast(col_type))

checkpoint_window = Window.partitionBy("portal").orderBy(
    F.col("checkpoint_updated_at").desc_nulls_last(),
    F.col("ingestion_timestamp").desc_nulls_last(),
    F.col("checkpoint_total_scraped").desc_nulls_last(),
)

df_checkpoints = (
    df_checkpoints_silver
    .select(
        "portal",
        "checkpoint_key",
        "checkpoint_source_path",
        "checkpoint_activo",
        "checkpoint_last_page",
        "checkpoint_total_scraped",
        "checkpoint_updated_at",
        "checkpoint_read_error",
        "ingestion_timestamp",
    )
    .withColumn("portal", F.lower(F.trim(F.col("portal"))))
    .filter(F.col("portal").isNotNull() & (F.col("portal") != ""))
    .withColumn("checkpoint_activo", F.coalesce(F.col("checkpoint_activo"), F.lit(0)))
    .withColumn("checkpoint_rank", F.row_number().over(checkpoint_window))
    .filter(F.col("checkpoint_rank") == 1)
    .drop("checkpoint_rank")
)

df_portal_operacion = (
    df_gold.groupBy("fuente")
    .agg(
        F.count("*").alias("portal_ofertas_activas"),
        F.expr("percentile_approx(precio_m2, 0.5)").alias("precio_m2_mediano_portal"),
        F.expr("percentile_approx(area_m2, 0.5)").alias("area_mediana_portal"),
        F.round(F.avg(F.coalesce(F.col("data_completeness"), F.lit(0.0))), 3).alias("data_completeness_promedio"),
        F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales_portal"),
        F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_portal_pct"),
        F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal_portal"),
        F.max("fecha_extraccion").alias("ultima_fecha_extraccion"),
    )
    .withColumnRenamed("fuente", "portal")
    .join(df_checkpoints, on="portal", how="left")
    .withColumn("checkpoint_prefix_detectado", F.lit(checkpoint_source_label))
    .withColumn("checkpoint_activo", F.coalesce(F.col("checkpoint_activo"), F.lit(0)))
    .withColumn(
        "checkpoint_age_hours",
        F.when(
            F.col("checkpoint_updated_at").isNotNull(),
            F.round((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.col("checkpoint_updated_at"))) / F.lit(3600.0), 1),
        )
    )
    .withColumn(
        "checkpoint_status",
        F.when(F.col("checkpoint_activo") == 0, F.lit("sin_checkpoint"))
         .when(F.col("checkpoint_updated_at").isNull(), F.lit("activo_sin_fecha"))
         .when(F.col("checkpoint_age_hours") <= 48, F.lit("activo_reciente"))
         .when(F.col("checkpoint_age_hours") <= 168, F.lit("activo_espera"))
         .otherwise(F.lit("activo_estancado"))
    )
    .withColumn(
        "portal_health_score",
        F.round(
            F.greatest(
                F.lit(0.0),
                F.least(
                    F.lit(100.0),
                    F.lit(55.0)
                    + F.least(F.log1p(F.col("portal_ofertas_activas")) * F.lit(8.0), F.lit(20.0))
                    + F.coalesce(F.col("pct_multiportal_portal"), F.lit(0.0)) * F.lit(0.10)
                    + (F.lit(100.0) - F.least(F.coalesce(F.col("dispersion_promedio_portal_pct"), F.lit(100.0)), F.lit(100.0))) * F.lit(0.10)
                    - F.when(F.col("checkpoint_activo") == 1, F.lit(15.0)).otherwise(F.lit(0.0))
                    - F.when(F.coalesce(F.col("checkpoint_age_hours"), F.lit(0.0)) > 168, F.lit(10.0)).otherwise(F.lit(0.0))
                ),
            ),
            1,
        ),
    )
    .withColumn("gold_snapshot_at", F.current_timestamp())
    .orderBy(F.asc("checkpoint_activo"), F.desc("portal_health_score"), F.desc("portal_ofertas_activas"))
)

print("\n🛰️ Analítica operativa por portal")
if checkpoint_source_label:
    print(f"   Checkpoints leídos desde Silver: {checkpoint_source_label}")
else:
    print("   No hay snapshot limpio de checkpoints en Silver. Ejecuta Silver antes de Gold.")
print(f"   Checkpoints útiles consolidados: {df_checkpoints.count():,}")
print(f"   Portales monitoreados: {df_portal_operacion.count()}")
display(df_portal_operacion.limit(100))

In [0]:
# =============================================================
# CELDA 3: GUARDAR — Gold Resumen (Delta) + Gold Sectorial (Delta) + Gold Analítica (Delta) + Gold Portal Ops (Delta) + Gold Detalle (Parquet)
# =============================================================

# Gold Resumen: agregación robusta por zona
ruta_resumen = f"s3a://{BUCKET}/gold/mercado_resumen/"
writer = df_resumen.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_resumen)
print(f"💾 Gold resumen: {ruta_resumen}")

# Gold Sectorial: tabla reutilizable para analítica, app y serving
ruta_sectorial = f"s3a://{BUCKET}/gold/mercado_sectorial/"
writer = df_sectorial.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_sectorial)
print(f"🧭 Gold sectorial: {ruta_sectorial}")

# Gold Analítica: bounds jerárquicos y score de calidad para limpieza y fallback
ruta_analytics = f"s3a://{BUCKET}/gold/mercado_analitica/"
writer = df_market_analytics.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_analytics)
print(f"📐 Gold analítica: {ruta_analytics}")

# Gold Portal Ops: estado operativo por fuente a partir de checkpoints y snapshot Gold
ruta_portal_ops = f"s3a://{BUCKET}/gold/portal_operacion/"
writer = df_portal_operacion.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_portal_ops)
print(f"🛰️ Gold portal ops: {ruta_portal_ops}")

# Gold Detalle: inmuebles limpios con KPIs robustos
ruta_app = f"s3a://{BUCKET}/gold/app_inmuebles/"
writer = df_gold.coalesce(1).write.format("parquet").mode("overwrite")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_app)
print(f"🚀 Gold detalle: {ruta_app}")